
# Integrated Audio Database Preparation and Analysis Notebook

This notebook combines functionalities from the `archon_split`, `archon_dedupe`, `archon_analyze`, and `archon_post_process` scripts.
It provides a streamlined workflow for:
1. Splitting audio into homogeneous chunks.
2. Deduplicating files to remove redundant material.
3. Analyzing audio descriptors and optionally reorganizing directories.
4. Post-processing data for insights (e.g., histograms of descriptors).

## Recommended Flow
1. **Run the Split Module**: Process a directory of audio files into smaller chunks.
2. **Run the Deduplication Module**: Remove duplicate files from the processed directory.
3. **Run the Analysis Module**: Extract descriptors and optionally sort files based on these descriptors.
4. **Run the Post-Processing Module**: Visualize and analyze the results.


## Split Module

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pydub
from pydub import AudioSegment
import math
import os, os.path



In [ ]:
destination_db = "/content/drive/My Drive/Scammers_Sorted_PM" #@param {type:"string"}
directory_db = "/content/drive/My Drive/Archon_Scammers" #@param {type:"string"}
slice_size = "250" #@param {type:"string"}
index_num = "5" #@param {type:"string"}

slice_size = int(slice_size)
counter = 0
index_num = int(index_num)



destination_db: storage location for new sliced files

directory_db: location of directory to scrape for audio

slice size: in ms

index_num: number of characters to use to sort slice db

In [ ]:
class SplitWav():
    def __init__(self, filename, destination):
        self.dest = destination
        self.filename = filename
        
        self.audio = AudioSegment.from_wav(self.filename)
    
    def get_duration(self):
        return self.audio.duration_seconds
    
    def single_split(self, from_msec, to_msec, split_filename):
        t1 = from_msec
        t2 = to_msec
        split_audio = self.audio[t1:t2]
        sort_dir = self.dest + '/' + split_filename[:index_num] + '/'
        if (os.path.exists(sort_dir) == False): os.makedirs(sort_dir)
        split_audio.export(sort_dir + split_filename, format="wav")
        
    def multiple_split(self, filebase, msec_per_split):
        total_sec = int(self.get_duration()) * 1000
        for i in range(0, total_sec, msec_per_split):
            filebase = filebase.rsplit( ".", 1)[0]
            split_fn = str(filebase) + "_" + str(i) + ".wav"
            self.single_split(i, i + msec_per_split, split_fn)

In [ ]:
for filename in os.scandir(directory_db):
  if (filename.path.endswith(".wav")):  
    split_wav = SplitWav(filename, destination_db)
    split_wav.multiple_split(filename.name, msec_per_split = slice_size)
    counter += 1
    print("completed " + str(counter) + " of " + str(dir_datasize))

In [ ]:
    drive.flush_and_unmount()   


## Deduplication Module

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from collections import defaultdict
import hashlib
import os
import sys

In [ ]:
destination_db = "/content/drive/My Drive/IRCMS_GAN_collaborative_database/Experiments/colab-violingan/archon-analysis" #@param {type:"string"}
des_datasize = "if known" #@param {type:"string"}
min_size = "176440" #@param {type:"string"}

min_size = int(min_size)


In [ ]:
# OPTIONAL
# dest_datasize = len(
    # [name for name in os.listdir(destination_db) if os.path.isfile(
        # os.path.join(destination_db, name))])
# NOTE: depending on size of folder, this can error out multiple times - 
#Colab will cache the results, so retry until success and log the number for future attempts.
# print(dest_datasize)

In [ ]:
def chunk_reader(fobj, chunk_size=1024):
    while True:
        chunk = fobj.read(chunk_size)
        if not chunk:
            return
        yield chunk


def get_hash(filename, first_chunk_only=False, hash=hashlib.sha1):
    hashobj = hash()
    file_object = open(filename, 'rb')

    if first_chunk_only:
        hashobj.update(file_object.read(1024))
    else:
        for chunk in chunk_reader(file_object):
            hashobj.update(chunk)
    hashed = hashobj.digest()

    file_object.close()
    return hashed


def check_for_duplicates(path, hash=hashlib.sha1):
    hashes_by_size = defaultdict(list)  # dict of size_in_bytes: [full_path_to_file1, full_path_to_file2, ]
    hashes_on_1k = defaultdict(list)  # dict of (hash1k, size_in_bytes): [full_path_to_file1, full_path_to_file2, ]
    hashes_full = {}   # dict of full_file_hash: full_path_to_file_string
    counter = 0


    for filename in os.scandir(destination_db):
        counter += 1
        if (counter % 1000 == 0): print("completed " + str(counter) + " of " + str(dest_datasize))
        full_path = destination_db + "/" + str(filename.name)
        file_size = os.path.getsize(full_path)
        if (file_size < min_size): 
          print("removing" + filename)
          os.remove(filename)

        if (file_size >= min_size): hashes_by_size[file_size].append(full_path)

    counter = 0

    for size_in_bytes, files in hashes_by_size.items():

        if len(files) < 2:
            continue 

        for filename in files:
          counter += 1
          if (counter % 1000 == 0): print("completed hash for" + str(counter) + " of " + str(len(hashes_by_size)))
          small_hash = get_hash(filename, first_chunk_only=True)
          hashes_on_1k[(small_hash, size_in_bytes)].append(filename)

    for __, files_list in hashes_on_1k.items():
        if len(files_list) < 2:
            continue

        for filename in files_list:
            try: 
                full_hash = get_hash(filename, first_chunk_only=False)
                duplicate = hashes_full.get(full_hash)
                if duplicate:
                    print("Duplicate found: {} and {}".format(filename, duplicate))
                    os.remove(filename)
                else:
                    hashes_full[full_hash] = filename
            except (OSError,):
                # the file access might've changed till the exec point got here 
                continue


if __name__ == "__main__":
      check_for_duplicates("/content/drive/My Drive/IRCMS_GAN_collaborative_database/Experiments/colab-violingan/archon-analysis")

## Analysis Module

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
!pip install librosa==0.8.0
import numpy as np
import json as json
import librosa
import os, os.path

In [ ]:
destination_db = "/content/drive/My Drive/Scammers_Sorted_PM" #@param {type:"string"}
dest_datasize = "10" #@param {type:"string"}
slice_size = "2" #@param {type:"string"}
hop_length = "2048" #@param {type:"string"}
sr = "44100" #@param {type:"string"}
index_num = "5" #@param {type:"string"}
output_filename = "/content/drive/My Drive/analysis_250ms.json" #@param {type:"string"}
analysis_filename = "/content/drive/My Drive/analysis_desilenced.json" #@param {type:"string"}

hop = int(hop_length)
sr = int(sr)
slice_size = int(slice_size)
index_num = int(index_num)
dest_datasize = int(dest_datasize)


In [ ]:
## STORE ANAYLSIS AS JSON

def export_to_json (data, savefile = output_filename):

  with open(savefile, 'a') as outfile:
    json.dump(data, outfile, indent=2)


## IMPORT JSON FILE

def json_load (filename):

  f = open(filename)
  l = json.load(f)
  return l

In [ ]:
## GRAB DESCRIPTORS FROM AUDIODB

def grab_descriptors(filename, sr = sr):
    
  y, sr = librosa.load(filename, sr = sr)

  cent = np.median(
    np.ndarray.flatten(
    librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)))
  flat = np.median(
    np.ndarray.flatten(
    librosa.feature.spectral_flatness(y=y, hop_length=hop)))
  rolloff = np.median(
    np.ndarray.flatten(
    librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=hop)))
  rms = np.median(
    np.ndarray.flatten(
    librosa.feature.rms(y=y, hop_length=hop)))
    
  f0, voiced_flag, voiced_probs = librosa.pyin(y,
                                fmin=librosa.note_to_hz('C2'),
                                fmax=librosa.note_to_hz('C7'))
    
  voiced = np.median(np.ndarray.flatten(voiced_flag))

  if (voiced == True):
    f0 = f0[~np.isnan(f0)]
    pitch = np.median(np.ndarray.flatten(f0))
    pitch = str(librosa.hz_to_note(pitch))
    pitch = pitch.replace("♯", "#") 

  else: 
    pitch = "unpitched"

  dict_ = { 
        "cent": str(cent),
        "flat": str(flat),
        "rolloff": str(rolloff),
        "rms": str(rms),
        "pitch": pitch
        }
  
  return dict_


def analyze_db(db = destination_db, samplerate = 44100, outfile = output_filename):

  dict_ = {}
  print ("...looking for previous progress... WARN: this can take VERY LONG depending on directory size & layout!!")

  if (os.path.exists(output_filename)): 
    print("...loading previous progress...")
    dict_ = json_load(output_filename)
    skiplen = len(dict_.keys())
    print("OK: done, found " + str(skiplen) + " entries.")
  else:
    print ("WARN: no previous progress found.")
    
  counter = 0
  skipcounter = 0

  for element in os.scandir(db):
    if element.is_dir():
      for entry in os.scandir(element):
        if (entry.path.endswith(".wav")):
          if (str(entry.name) in dict_.keys()): skipcounter += 1
          else:
            try: 
              dict_[str(entry.name)] = grab_descriptors(entry)
            except: print ("ERR: file " + str(entry.name) + " could not be processed.")
      
        counter += 1
        if ((counter % 5000) == 0): 

          print("OK: " + str(counter) + " completed.")
          if (skipcounter > 0 & skipcounter < skiplen): print("OK: skipped " + str(skipcounter) + " files that have already been processed.")
          
          if (skipcounter != counter):
            print("...saving... ")
            if (os.path.exists(output_filename)): os.remove(output_filename)
            export_to_json(dict_, output_filename)
            print("OK: saved!")
          else: print("OK: still processing skipped files, no need to save.")

  if (os.path.exists(output_filename)): os.remove(output_filename)
  export_to_json(dict_, output_filename)
  print("OK: successfully completed!")


In [ ]:
## TO ANALYZE DB AND EXPORT TO JSON
complete_analysis = analyze_db(destination_db)

In [ ]:
def strip_silence (dict_):

  counter = 0

  for k, v in dict_.copy().items():
    sample = v
    filename = k
    if (float(sample.get("rms")) < 0.01):
      dict_.pop(filename, None)
      if (os.path.exists(destination_db + '/' + filename[:index_num] + '/' + filename)): os.remove(destination_db + '/' + filename[:index_num] + '/' + filename)
      counter = counter + 1
      print( "removed " + str(filename) + ", " + str(counter) + " items removed.")

  export_to_json(dict_, analysis_filename)


In [ ]:
strip_silence (json_load(output_filename))
print("OK: desilenced")
drive.flush_and_unmount()
print("OK: successfully unmounted VM!")

In [ ]:
## FIRST PASS MAY TAKE A BIT AND/OR ERROR OUT IF DIR IS HUGE - JUST BE PATIENT AND RETRY, IT WILL GO THROUGH EVENTUALLY

def sort_by_pitch (unsort_or_sort, dict_):

  counter = 0 
  pitch = ""
  print ("...loading directory... WARN: this can take VERY LONG depending on directory size & layout!!")
  
  for k, v in dict_.items():

    sample = v
    filename = k
    counter += 1

    pitch = sample.get("pitch")
    
    if (pitch != "unpitched"): 
      oct = int(pitch[-1])
      pitch = pitch.replace(str(oct), "")  
      pitch_dir = destination_db + "/" + pitch + "/" + str(oct) + "/"   

    else: 
      pitch_dir = destination_db + "/" + pitch + "/"
      cent = sample.get("cent")
      flat = sample.get("flat")
      rolloff = sample.get("rolloff")

      if float(cent) > 4000.0: pitch_dir = pitch_dir + "high_cent/"
      else: pitch_dir = pitch_dir + "low_cent/"

      if float(flat) > 0.01: pitch_dir = pitch_dir + "high_flat/"
      else: pitch_dir = pitch_dir + "low_flat/"

      if float(rolloff) > 8000: pitch_dir = pitch_dir + "high_rolloff/"
      else: pitch_dir = pitch_dir + "low_rollof/"     

    if (unsort_or_sort == "sort"):     
      if (os.path.exists(pitch_dir) == False): os.makedirs(pitch_dir)
      if (os.path.exists(pitch_dir + filename) == False): os.replace(destination_db + '/' + filename[:index_num] + '/' + filename, pitch_dir + filename)
      else: print ("OK: File " + pitch_dir + filename + " has already been moved.")

    else: 
      if (os.path.exists(pitch_dir + filename) == True): 
        os.rename(pitch_dir + filename, destination_db + filename)
        print(destination_db + filename)
    
    if (counter == 1): print("OK: beginning to sort.")
    if ((counter % 5000) == 0): print("OK: processed " + str(counter) + " files.")
  
  print("OK: successfully completed! WARN: you need to flush and unmount the VM for changes to fully take effect!")

In [ ]:
## TO MOVE FILES BY PITCH INTO SUBDIRECTORIES
sort_by_pitch ("sort", json_load(analysis_filename))
print("OK: sorted")
drive.flush_and_unmount()
print("OK: successfully unmounted VM!")

## Post-Processing Module

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import json as json
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
analysis_filename = "/content/drive/My Drive/analysis_500ms.json" #@param {type:"string"}


In [ ]:
## IMPORT JSON FILE
def json_load (filename):
  f = open(filename)
  l = json.load(f)
  return l

In [ ]:
## COLLECT STATS
def unpack_json(dict, metric = "cent", pitched = "both"):

  data = []

  for k, v in d_lib.items():
    filename = k
    sample = v

    if (pitched == "both"):
      if (metric == "pitch"): 
        if sample.get(metric) != "unpitched": 
          pitch = str(sample.get("pitch"))
          data.append(pitch[:-1])
        else: data.append(sample.get(metric))
      else: data.append(float(sample.get(metric)))

    elif (pitched == "pitched"):
      if (sample.get("pitch") != "unpitched"):
        if (metric == "pitch"): data.append(sample.get(metric))
        else: data.append(float(sample.get(metric)))

    elif (pitched == "unpitched"):
      if (sample.get("pitch") == "unpitched"):
        if (metric == "pitch"): 
          print ("ERROR - check function args")
          break
        else: data.append(float(sample.get(metric)))

  return data

In [ ]:
def histogram(dict_, metric, pitched, min, max):
  plt.rcParams["figure.figsize"] = [17.50, 11]
  plt.rcParams["figure.autolayout"] = True

  print("starting graph")
  array_ = np.sort(
          np.array(
              unpack_json(
                  dict_, metric, pitched)))
  bins_ = np.linspace(min, max, 100)

  plt.title((metric + " Distribution"))
  plt_ = plt.hist(array_, bins_)
 
  plt.xlabel(metric)
  plt.ylabel('Magnitude')

  plt.show()      

In [ ]:
d_lib = json_load(analysis_filename)

In [ ]:
histogram(d_lib, "cent", "unpitched", 0, 10000)
#pitched < 4k, peak around 2.2k
#unpitched >2.2k, peak around 3.2k, goes until 8k or so

In [ ]:
histogram(d_lib, "flat", "unpitched", 0, 0.1)
## unpitched flatness is much greater generally than pitched flatness

In [ ]:
histogram(d_lib, "rolloff", "unpitched", 0, 18000)
#pitched < 7.5k generally, peak around 4-5k
# unpitched 4k-15k, peaks throughout, much of the center of gravity between 4k and 12.5k

In [ ]:
histogram(d_lib, "rms", "both", 0, 0.5)

In [ ]:
x = pd.Series(
        unpack_json(
            d_lib, "pitch", "both")
        ).value_counts()

#x
x.plot(kind='bar')
## most information is nonpitched, though about 1/6th of the elements are pitched 